In [27]:
import sys
from pathlib import Path
from dotenv import load_dotenv
# Production layout: add project root and src for imports (run from repo root or notebooks/ingestion/)
_root = Path(".").resolve()
if _root.name == "ingestion":
    _root = _root.parent.parent
elif (_root / "src").is_dir():
    pass
else:
    _root = _root.parent
load_dotenv(_root / ".env")
load_dotenv("/app/.env")
sys.path.insert(0, str(_root))
sys.path.insert(0, str(_root / "src"))

from storage.postgres.pgConn import PgConn
from storage.postgres import PostgresSQL_table_queries
from storage.postgres.news_dataframe import (
    filter_financial_news_by_date,
    filter_financial_news_ingested_today,
    filter_financial_news_published_today,
    get_financial_news_content_by_id,
    normalize_financial_news_datetime_column,
)
from storage.cloud.CloudStorage import CloudStorageProvider

import pandas as pd
from datetime import date, datetime

# Sanity check: if this fails, use File → Reload Notebook from Disk, then restart kernel
import storage.postgres.news_dataframe as _news_df
print(f"Using news_dataframe from: {_news_df.__file__}")
print(f"filter_financial_news_ingested_today: OK")

Using news_dataframe from: /app/src/storage/postgres/news_dataframe.py
filter_financial_news_ingested_today: OK


In [28]:
table_name = PostgresSQL_table_queries.FINANCIAL_NEWS_TABLE_NAME
pg_conn = PgConn(table_name)
df = pg_conn.get_financial_news()
if df is None:
    raise RuntimeError(
        "get_financial_news() failed — check the error printed above "
        "(connection, table name, or schema)."
    )

Connection to the database successful!
Table name set to: financial_news_241118


In [29]:
print(f"Total news articles queried: {df.shape[0]}")

Total news articles queried: 90


In [30]:
df.head()

,id,source,headline,href,summary,content,author,minsread,datetime,created_at
0,687238651341839350,Motley Fool,"A Nationwide, Industry-Owned Blockchain Networ...",https://finance.yahoo.com/markets/crypto/artic...,,Even before the advent of cryptocurrencies and...,"Bram Berkowitz, The Motley Fool",4 min read,2026-08-29 09:50:00,2026-08-30 17:18:53.185133
1,2778964649665051367,BeInCrypto,American Insurers Secretly Put $16 Billion of ...,https://finance.yahoo.com/markets/stocks/artic...,,Delaware Life Insurance Company relabeled $16....,Lockridge Okoth,3 min read,2026-08-30 12:15:06,2026-08-30 17:18:52.980145
2,3837912206825908155,Stocktwits,Arthur Hayes Names Ethereum His ‘No. 1 Crypto ...,https://finance.yahoo.com/markets/crypto/artic...,,An Ethereum whale linked to OTC crypto desk BI...,Yashu Gola,3 min read,2026-08-28 12:15:50,2026-08-30 17:18:53.116535
3,4501225223203342451,Euronews,As bitcoin soars people ask who is Giancarlo D...,https://finance.yahoo.com/markets/crypto/artic...,,Bitcoin is back in the spotlight after a week-...,Stefania De Michele,5 min read,2026-08-29 15:25:21,2026-08-30 17:18:53.081195
4,1339205561091018351,Motley Fool,Billionaire Hedge Fund Investor Ray Dalio Now ...,https://finance.yahoo.com/markets/crypto/artic...,,"Ray Dalio, founder of Bridgewater Associates, ...","Lyle Daly, The Motley Fool",4 min read,2026-08-29 05:20:00,2026-08-30 17:18:53.200538


In [31]:
def delete_records_for_current_date(df, pg_conn):
    if df is None:
        print("DataFrame is empty. No records to delete.")
        return
        
    today_records = filter_financial_news_by_date(df)

    # Extract date strings (DB/delete API expects stored string form)
    date_strings = today_records['datetime'].astype(str).tolist()

    # Call the delete_records_by_date method
    pg_conn.delete_records_by_date(date_strings)

# Then call the delete_records_for_current_date method
#delete_records_for_current_date(df, pg_conn)
#list_ids = ["11111"]
#pg_conn.delete_records_by_ids(list_ids)

In [32]:
class DataETL():
    
    def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
    class Export():
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def set_dataframe(self, dataframe):
            self.df = dataframe
        
        def export_text_to_s3(self, bucket_name, prefix_path, file_format):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()

            # Create a new bucket
            aws_storage.create_bucket(bucket_name)

            # Upload DataFrame with datetime subfolder structure
            aws_storage.upload_dataframe_with_datetime_subfolders(self.df, bucket_name, prefix_path, file_format)
        
        def export_text_to_s3_full_file(self, bucket_name, prefix_path, filename):
            aws_storage = self.cloudProvider.AWS()
            aws_storage.upload_dataframe_to_csv(self.df, bucket_name, filename, prefix_path)
            
    class Ingestion():
        def __init__(self, dataframe):
            self.df = dataframe
            self.cloudProvider = CloudStorageProvider()
            
        def get_full_data_csv_file(self, bucket_name, prefix_path):
            # Initialize AWS storage
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_csv_from_specific_folder(bucket_name, prefix_path)
        
        def get_data_csv_file_by_datetime(
            self, bucket_name, prefix_path, year, month, day, hour=None, minute=None, second=None
        ):
            aws_storage = self.cloudProvider.AWS()
            return aws_storage.get_dataframe_from_specific_datetime(
                bucket_name,
                prefix_path,
                year=year,
                month=month,
                day=day,
                hour=hour,
                minute=minute,
                second=second,
            )
    
    class Process():
        
        def __init__(self, dataframe):
            self.df = dataframe
        
        def getData():
            self.df = pg_conn.get_financial_news()
        
        def filter_by_current_date(self):
            # Article publish date (datetime) on today's calendar date (default on_date=None)
            return filter_financial_news_by_date(self.df)
            
    class Transform():
        def extractStopWords():
            pass

In [33]:
etl = DataETL(df)

# Export by article publish time (datetime). on_date=None → today's local date (date.today()).
export_on_date = "2026-08-30"  # e.g. "2026-05-23" to override today
filtered_df = filter_financial_news_by_date(df, on_date=export_on_date)

print(
    f"Filter date (datetime column): {export_on_date or date.today()} | "
    f"Rows matched: {len(filtered_df)} | "
    f"Ingested today (created_at only): {len(filter_financial_news_ingested_today(df))}"
)
filtered_df.head()

Filter date (datetime column): 2026-08-30 | Rows matched: 16 | Ingested today (created_at only): 90


,id,source,headline,href,summary,content,author,minsread,datetime,created_at
1,2778964649665051367,BeInCrypto,American Insurers Secretly Put $16 Billion of ...,https://finance.yahoo.com/markets/stocks/artic...,,Delaware Life Insurance Company relabeled $16....,Lockridge Okoth,3 min read,2026-08-30 12:15:06,2026-08-30 17:18:52.980145
7,1711274765437335400,Stocktwits,Bitcoin Buyers Step Back In As Realized Cap Ga...,https://finance.yahoo.com/markets/crypto/artic...,,On-chain data showed Bitcoin's realized cap in...,Anushka Basu,4 min read,2026-08-30 11:50:08,2026-08-30 17:18:52.983212
8,4492142120262490523,Motley Fool,"Bitcoin Could Hit $300,000 by 2030, Says Coinb...",https://finance.yahoo.com/markets/crypto/artic...,,Bitcoin (CRYPTO: BTC) rose 25% in August. That...,"Dominic Basulto, The Motley Fool",4 min read,2026-08-30 00:20:00,2026-08-30 17:18:53.030617
21,2175798569092616230,Motley Fool,Can Bitcoin Reach $1 Million by 2030? One Cryp...,https://finance.yahoo.com/markets/crypto/artic...,,With Bitcoin (CRYPTO: BTC) on a hot rally and ...,"Alex Carchidi, The Motley Fool",4 min read,2026-08-30 09:49:00,2026-08-30 17:18:53.013165
29,4521490560772217326,Motley Fool,Coinbase Is Adding Perpetual Futures to Its Ba...,https://finance.yahoo.com/markets/crypto/artic...,,"On Aug. 19, Coinbase Global (NASDAQ: COIN) lau...","Alex Carchidi, The Motley Fool",4 min read,2026-08-30 10:10:00,2026-08-30 17:18:53.008521


In [34]:
print(f"Total filtered news articles queried: {filtered_df.shape[0]}")

Total filtered news articles queried: 16


In [35]:
import os

export_rows_to_s3 = True
etl_export = etl.Export(filtered_df)
bucket_name = "test-financial-news-bucket"
prefix_path = "news/crypto"
file_format = "csv"

if export_rows_to_s3 and not filtered_df.empty:
    if not os.getenv("AWS_ACCESS_KEY_ID") or not os.getenv("AWS_SECRET_ACCESS_KEY"):
        raise RuntimeError(
            "AWS credentials not configured. Uncomment and set AWS_ACCESS_KEY_ID and "
            "AWS_SECRET_ACCESS_KEY in .env, then restart Jupyter: ./docker/start_jupyter.ps1"
        )
    export_df = normalize_financial_news_datetime_column(filtered_df)
    etl_export.set_dataframe(export_df)
    etl_export.export_text_to_s3(bucket_name, prefix_path, file_format)

Bucket 'test-financial-news-bucket' already exists.
Data for row 1 with id '2778964649665051367' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=08/day=30/hour=12/minute=15/second=06/format=csv/2778964649665051367.csv'
Data for row 7 with id '1711274765437335400' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=08/day=30/hour=11/minute=50/second=08/format=csv/1711274765437335400.csv'
Data for row 8 with id '4492142120262490523' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=08/day=30/hour=00/minute=20/second=00/format=csv/4492142120262490523.csv'
Data for row 21 with id '2175798569092616230' uploaded to S3 bucket 'test-financial-news-bucket' under folder 'news/crypto/year=2026/month=08/day=30/hour=09/minute=49/second=00/format=csv/2175798569092616230.csv'
Data for row 29 with id '4521490560772217326' uploaded to S3 bucket 'test-financial-news-bucket' un

In [36]:
post_full_csv = False
if (post_full_csv == True) and not filtered_df.empty:
    now = datetime.now()
    filename = f"{now.year}-{now.month:02}-{now.day:02}_full_record"
    etl_export.set_dataframe(etl.df)
    etl_export.export_text_to_s3_full_file(bucket_name, prefix_path, filename)

In [37]:
ingest_data = False
get_full_file = False
get_by_datetime = True
df_from_file = None

if ingest_data == True:
    etl_ingestion = etl.Ingestion(etl.df)
    bucket_name = "test-financial-news-bucket"
    prefix_path = "news/crypto/"
    year = '2026'
    month = '05'
    day = '24'
    hour = None   # set e.g. '04' to narrow to one hour; None = whole day
    minute = None
    if get_full_file == True:
        filename = f"{year}-{month}-{day}_full_record.csv"
        full_path = f"{prefix_path}{filename}"
        df_from_file = etl_ingestion.get_full_data_csv_file(bucket_name, full_path)
    elif get_by_datetime == True:
        df_from_file = etl_ingestion.get_data_csv_file_by_datetime(bucket_name, prefix_path, year, month, day, hour, minute)

In [38]:
if df_from_file is not None:
    print(df_from_file.count())
    df_from_file.head()

In [39]:
targetId = ""  # i.e. "1221589746717124508" targetId can be string or numeric type

# Lookup order: S3 ingest result, filtered export batch, then full DB pull
lookup_df = None
lookup_source = None
for name, candidate in (
    ("df_from_file", df_from_file if "df_from_file" in dir() else None),
    ("filtered_df", filtered_df if "filtered_df" in dir() else None),
    ("df", df if "df" in dir() else None),
):
    if candidate is not None and not getattr(candidate, "empty", True):
        lookup_df = candidate
        lookup_source = name
        break

if targetId and lookup_df is not None:
    full_content = get_financial_news_content_by_id(lookup_df, targetId)
    if full_content:
        print(full_content)
    else:
        print(
            f"No content for id {targetId!r} in {lookup_source} "
            f"({len(lookup_df)} rows). Id column dtype: {lookup_df['id'].dtype}"
        )
else:
    print("Set targetId and ensure df_from_file, filtered_df, or df is loaded.")

Set targetId and ensure df_from_file, filtered_df, or df is loaded.
